# 实验 2：6 位镜像对称

## 任务理解

输入是 6 位二进制向量。仅当

$$x_1=x_6,\quad x_2=x_5,\quad x_3=x_4$$

同时成立时，标签为 1，否则为 0。训练集包含全部 $2^6=64$ 种输入。

## 理论实现

定义三组镜像差

$$d_1=x_1-x_6,\quad d_2=x_2-x_5,\quad d_3=x_3-x_4.$$

网络采用 `6 -> 2 -> 1`：两个隐藏单元分别检测加权差的正、负方向，输出层在两个检测器都未激活时判断为对称。各层使用 Sigmoid，损失为半平方误差，参数用全批量动量梯度下降更新。

## 模型代码

In [1]:
from itertools import product

import numpy as np

CONFIG = {
    "seed": 1,
    "epochs": 1_425,
    "learning_rate": 0.1,
    "momentum": 0.9,
    "init_scale": 0.3,
}

X_train = np.array(list(product([0.0, 1.0], repeat=6)), dtype=np.float64)
y_train = np.all(X_train == X_train[:, ::-1], axis=1).astype(np.float64)[:, None]


def sigmoid(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    output = np.empty_like(x)
    positive = x >= 0
    output[positive] = 1.0 / (1.0 + np.exp(-x[positive]))
    exp_x = np.exp(x[~positive])
    output[~positive] = exp_x / (1.0 + exp_x)
    return output


class MirrorNetwork:
    def __init__(self, seed: int) -> None:
        rng = np.random.default_rng(seed)
        scale = CONFIG["init_scale"]
        self.params = {
            "W1": rng.uniform(-scale, scale, size=(6, 2)),
            "b1": rng.uniform(-scale, scale, size=(2,)),
            "W2": rng.uniform(-scale, scale, size=(2, 1)),
            "b2": rng.uniform(-scale, scale, size=(1,)),
        }
        self.velocity = {
            name: np.zeros_like(value) for name, value in self.params.items()
        }

    def forward(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=np.float64)
        if X.ndim != 2 or X.shape[1] != 6:
            raise ValueError(f"输入形状应为 (样本数, 6)，实际为 {X.shape}")
        hidden = sigmoid(X @ self.params["W1"] + self.params["b1"])
        return sigmoid(hidden @ self.params["W2"] + self.params["b2"])

    def train_step(self, X: np.ndarray, y: np.ndarray) -> float:
        hidden = sigmoid(X @ self.params["W1"] + self.params["b1"])
        output = sigmoid(hidden @ self.params["W2"] + self.params["b2"])

        delta_output = (output - y) * output * (1.0 - output)
        grads = {
            "W2": hidden.T @ delta_output,
            "b2": delta_output.sum(axis=0),
        }
        delta_hidden = (
            delta_output @ self.params["W2"].T
        ) * hidden * (1.0 - hidden)
        grads["W1"] = X.T @ delta_hidden
        grads["b1"] = delta_hidden.sum(axis=0)

        for name in self.params:
            self.velocity[name] = (
                CONFIG["momentum"] * self.velocity[name]
                - CONFIG["learning_rate"] * grads[name]
            )
            self.params[name] += self.velocity[name]

        return float(0.5 * np.sum((output - y) ** 2))

In [2]:
mirror_model = MirrorNetwork(CONFIG["seed"])
for _ in range(CONFIG["epochs"]):
    mirror_model.train_step(X_train, y_train)

train_probabilities = mirror_model.forward(X_train)
train_predictions = (train_probabilities >= 0.5).astype(int)
assert np.array_equal(train_predictions, y_train.astype(int))

## 测试结果

只需修改下面的 `test_inputs`。每一行必须包含 6 个 0/1。

In [3]:
# 修改这里的输入数据
test_inputs = np.array([
    [0, 1, 1, 1, 1, 0],
    [1, 0, 1, 1, 0, 1],
    [0, 1, 0, 1, 0, 1],
    [1, 0, 1, 0, 1, 0],
    [1, 1, 1, 1, 1, 0],
], dtype=np.float64)

if not np.all((test_inputs == 0) | (test_inputs == 1)):
    raise ValueError("测试输入只能包含 0 或 1")

expected = np.all(test_inputs == test_inputs[:, ::-1], axis=1).astype(int)
probabilities = mirror_model.forward(test_inputs).ravel()
predictions = (probabilities >= 0.5).astype(int)

print(f"训练集准确率: {(train_predictions == y_train).mean():.2%}")
print("输入 -> 正确标签 -> 对称概率 -> 预测标签")
for row, target, probability, prediction in zip(
    test_inputs.astype(int), expected, probabilities, predictions
):
    bits = "".join(map(str, row))
    print(f"{bits} -> {target} -> {probability:.6f} -> {prediction}")

训练集准确率: 100.00%
输入 -> 正确标签 -> 对称概率 -> 预测标签
011110 -> 1 -> 0.969008 -> 1
101101 -> 1 -> 0.969034 -> 1
010101 -> 0 -> 0.025408 -> 0
101010 -> 0 -> 0.025431 -> 0
111110 -> 0 -> 0.000376 -> 0
